In [ ]:
# SINGLE CONFIGURATION BLOCK — the pre-registered full analysis is the default.
RUN_MODE = "full"  # "full" or "smoke"
SOURCE_BUNDLE_URL = (
    "https://raw.githubusercontent.com/grewalsk/"
    "counterfactual-faithfulness-research/main/results/bundles/"
    "stage3b_result_bundle.zip"
)
LOCAL_SOURCE_BUNDLE = ""  # Optional absolute Colab path; URL is used when empty.
OUTPUT_DIR = "/content/stage4_matched_intervention"
SEED = 71
SEVERITIES = [0.00, 0.25, 0.50, 0.75, 1.00]
INTERVENTION_SEEDS = [1103, 2203, 3301, 4409, 5501]
BOOTSTRAP_REPS = 2000
EXPECTED_ACTIONS = 10
MATCH_TOLERANCE = 1e-10
DOWNLOAD_RESULTS = True

assert RUN_MODE in {"full", "smoke"}
assert SEVERITIES == [0.00, 0.25, 0.50, 0.75, 1.00]
assert len(INTERVENTION_SEEDS) == 5
assert len(set(INTERVENTION_SEEDS)) == len(INTERVENTION_SEEDS)
assert EXPECTED_ACTIONS == 10
assert BOOTSTRAP_REPS >= 2000


# Stage 4: matched-error action-structure intervention

Stage 3B showed that a frozen linear physical-state readout plans better than
raw latent distance in PushT and Wall, but its per-instance counterfactual
error did not improve held-out failure prediction.

Stage 4 tests the missing mechanism directly. It applies two interventions of
exactly matched magnitude to the untouched Stage 3B final-test predictions:

1. **action-structure corruption** destroys which predicted consequence
   belongs to which candidate action while preserving the state-level
   centroid and the empirical residual set;
2. **common-mode corruption** moves every predicted candidate consequence by
   the same amount.

If action-specific predictive structure matters for planning, the first
intervention must cause more regret and ranking damage than the second in
both simulators. The protocol and decision gate were frozen before evaluating
these intervention outcomes.


In [ ]:
# Runtime setup, source-bundle acquisition, and failure-safe output handling.
from __future__ import annotations

import hashlib
import json
import math
import shutil
import traceback
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = Path(OUTPUT_DIR)
SOURCE_DIR = OUT / "stage3b_source"
PLOT_DIR = OUT / "plots"
RESULT_ZIP = Path("/content/stage4_result_bundle.zip")

if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
(OUT / "FAILURE_TRACE.txt").write_text("NONE\n")

PIPELINE_FAILED = False


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, allow_nan=True) + "\n")


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def record_failure(label):
    global PIPELINE_FAILED
    PIPELINE_FAILED = True
    payload = f"STAGE: {label}\n\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(payload)
    print(payload)


STAGE4_CONFIG = {
    "run_mode": RUN_MODE,
    "source_bundle_url": SOURCE_BUNDLE_URL,
    "seed": SEED,
    "severities": SEVERITIES,
    "intervention_seeds": INTERVENTION_SEEDS,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "expected_actions": EXPECTED_ACTIONS,
    "match_tolerance": MATCH_TOLERANCE,
    "protocol": "matched decoded-pose error intervention",
    "source_split": "Stage 3B untouched final_test",
}
write_json(OUT / "stage4_config.json", STAGE4_CONFIG)

try:
    source_zip = OUT / "stage3b_result_bundle.zip"
    local_source = Path(LOCAL_SOURCE_BUNDLE) if LOCAL_SOURCE_BUNDLE else None
    if local_source is not None and local_source.is_file():
        shutil.copy2(local_source, source_zip)
    else:
        print(f"Downloading frozen Stage 3B source bundle from {SOURCE_BUNDLE_URL}")
        urllib.request.urlretrieve(SOURCE_BUNDLE_URL, source_zip)
    with zipfile.ZipFile(source_zip) as handle:
        handle.extractall(SOURCE_DIR)

    required_source_files = [
        "FAILURE_TRACE.txt",
        "metrics_summary.json",
        "stage3b_decision.json",
        "action_predictions.csv",
        "tasks.json",
        "result_zip_manifest.json",
    ]
    missing_source = [
        name for name in required_source_files
        if not (SOURCE_DIR / name).is_file()
    ]
    if missing_source:
        raise RuntimeError(f"Stage 3B source bundle is missing {missing_source}")
    if (SOURCE_DIR / "FAILURE_TRACE.txt").read_text().strip() != "NONE":
        raise RuntimeError("Stage 3B source bundle contains a failure trace")
    stage3b_metrics = json.loads(
        (SOURCE_DIR / "metrics_summary.json").read_text()
    )
    if stage3b_metrics.get("status") != "SUCCESS":
        raise RuntimeError("Stage 3B source metrics do not report SUCCESS")

    source_manifest = {
        "archive_sha256": sha256_file(source_zip),
        "archive_size_bytes": source_zip.stat().st_size,
        "required_files": required_source_files,
        "stage3b_status": stage3b_metrics["status"],
        "stage3b_decision": json.loads(
            (SOURCE_DIR / "stage3b_decision.json").read_text()
        ).get("status"),
        "source_final_test_only": True,
    }
    write_json(OUT / "source_bundle_manifest.json", source_manifest)
except Exception:
    record_failure("source_bundle")


In [ ]:
# Frozen metric, task-cost, intervention, and state-cluster bootstrap helpers.
RANKING_TIE = 1e-9


def pair_indices(length):
    return np.triu_indices(length, k=1)


def decoded_task_cost(environment, prediction, task):
    prediction = np.asarray(prediction, dtype=np.float64)
    if environment == "PushT":
        goal = np.asarray(task["goal"], dtype=np.float64)
        angle = np.arctan2(prediction[..., 2], prediction[..., 3])
        angle_error = np.arctan2(
            np.sin(angle - goal[2]),
            np.cos(angle - goal[2]),
        )
        return np.linalg.norm(
            np.concatenate(
                [
                    prediction[..., :2] - goal[:2] / 512.0,
                    (angle_error / np.pi)[..., None],
                ],
                axis=-1,
            ),
            axis=-1,
        )
    return np.linalg.norm(
        prediction[..., :2]
        - np.asarray(task["goal"], dtype=np.float64) / 65.0,
        axis=-1,
    )


def ranking_metrics(true_cost, predicted_cost, tie=RANKING_TIE):
    truth = np.asarray(true_cost, dtype=np.float64)
    prediction = np.asarray(predicted_cost, dtype=np.float64)
    selected = int(np.argmin(prediction))
    oracle = int(np.argmin(truth))
    best = float(np.min(truth))
    chosen = float(truth[selected])
    spread = float(np.max(truth) - best)
    regret = chosen - best
    normalized_regret = regret / spread if spread > tie else 0.0
    left, right = pair_indices(len(truth))
    true_margin = truth[left] - truth[right]
    predicted_margin = prediction[left] - prediction[right]
    valid = np.abs(true_margin) > tie
    credit = np.full(len(left), np.nan)
    same = np.sign(true_margin) == np.sign(predicted_margin)
    credit[valid & same] = 1.0
    credit[valid & (np.abs(predicted_margin) <= tie)] = 0.5
    credit[valid & np.isnan(credit)] = 0.0
    weights = np.abs(true_margin)
    pairwise = float(np.nanmean(credit)) if np.any(valid) else float("nan")
    weighted = (
        float(np.nansum(weights * credit) / np.sum(weights[valid]))
        if np.any(valid)
        else float("nan")
    )
    margin_scale = (
        float(np.sqrt(np.mean(true_margin[valid] ** 2)))
        if np.any(valid)
        else 0.0
    )
    normalized_margin_rmse = (
        float(
            np.sqrt(
                np.mean(
                    (predicted_margin[valid] - true_margin[valid]) ** 2
                )
            )
            / margin_scale
        )
        if margin_scale > tie
        else float("nan")
    )
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": float(chosen <= best + tie),
        "regret": float(regret),
        "normalized_regret": float(normalized_regret),
        "pairwise_accuracy": pairwise,
        "weighted_pairwise_accuracy": weighted,
        "normalized_margin_rmse": normalized_margin_rmse,
    }


def stable_seed(*items):
    payload = "|".join(str(item) for item in items).encode()
    return int.from_bytes(
        hashlib.sha256(payload).digest()[:8], "little"
    ) % (2**32)


def fixed_derangement(length, seed):
    rng = np.random.default_rng(seed)
    identity = np.arange(length)
    for _ in range(1000):
        candidate = rng.permutation(length)
        if np.all(candidate != identity):
            return candidate
    return np.roll(identity, 1)


def matched_pose_interventions(predicted_pose, severity, seed):
    predicted_pose = np.asarray(predicted_pose, dtype=np.float64)
    count, dimension = predicted_pose.shape
    permutation = fixed_derangement(count, seed)
    centroid = np.mean(predicted_pose, axis=0)
    residual = predicted_pose - centroid
    full_action_delta = residual[permutation] - residual
    full_rms = float(
        np.sqrt(np.mean(np.sum(full_action_delta**2, axis=1)))
    )
    rng = np.random.default_rng(seed + 1)
    direction = rng.standard_normal(dimension)
    norm = float(np.linalg.norm(direction))
    if norm <= 1e-15:
        direction = np.zeros(dimension)
        direction[0] = 1.0
    else:
        direction = direction / norm
    action_pose = (
        centroid
        + residual
        + float(severity) * full_action_delta
    )
    common_pose = (
        predicted_pose
        + float(severity) * full_rms * direction[None, :]
    )
    action_rms = float(
        np.sqrt(
            np.mean(
                np.sum((action_pose - predicted_pose) ** 2, axis=1)
            )
        )
    )
    common_rms = float(
        np.sqrt(
            np.mean(
                np.sum((common_pose - predicted_pose) ** 2, axis=1)
            )
        )
    )
    return {
        "action_structure": action_pose,
        "common_mode": common_pose,
        "action_rms": action_rms,
        "common_rms": common_rms,
        "permutation": permutation,
        "fixed_points": int(np.sum(permutation == np.arange(count))),
    }


def bootstrap_mean(values, groups, repetitions, seed):
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups)
    finite = np.isfinite(values)
    values = values[finite]
    groups = groups[finite]
    unique = np.unique(groups)
    if not len(unique):
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
            "n_bootstrap": int(repetitions),
        }
    grouped = [values[groups == group] for group in unique]
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions, dtype=np.float64)
    for index in range(repetitions):
        sampled = rng.integers(0, len(unique), size=len(unique))
        draws[index] = np.mean(
            np.concatenate([grouped[item] for item in sampled])
        )
    return {
        "estimate": float(np.mean(values)),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "n_clusters": int(len(unique)),
        "n_bootstrap": int(repetitions),
    }


## Integrity and estimands

Intervention directions and derangements are deterministic functions of
pre-specified seeds and unit identifiers. Probe seeds, model families,
horizons, and intervention seeds are repeated measurements. The bootstrap
resamples only the 40 state clusters in each environment.

At full severity, each environment must have a positive state-clustered 95%
lower bound for both:

- `regret(action-structure) - regret(common-mode)`;
- `weighted-ranking(common-mode) - weighted-ranking(action-structure)`.

All matched pose-displacement discrepancies must be at most `1e-10`.


In [ ]:
# Load untouched Stage 3B final-test linear-pose rows and validate unit structure.
if not PIPELINE_FAILED:
    try:
        tasks_payload = json.loads((SOURCE_DIR / "tasks.json").read_text())
        task_lookup = {
            (item["environment"], int(item["task_id"])): item
            for item in tasks_payload
        }
        raw_actions = pd.read_csv(SOURCE_DIR / "action_predictions.csv")
        linear = raw_actions.loc[
            raw_actions["readout"].eq("linear_pose")
        ].copy()
        required_columns = {
            "environment",
            "state_id",
            "task_id",
            "model",
            "probe_seed",
            "horizon",
            "action",
            "true_cost",
            "predicted_cost",
            "predicted_pose_0",
            "predicted_pose_1",
        }
        missing_columns = sorted(required_columns - set(linear.columns))
        if missing_columns:
            raise RuntimeError(f"Missing source columns: {missing_columns}")
        if set(linear["environment"].unique()) != {"PushT", "Wall"}:
            raise RuntimeError("Expected exactly PushT and Wall")
        group_columns = [
            "environment",
            "state_id",
            "task_id",
            "model",
            "probe_seed",
            "horizon",
        ]
        action_counts = linear.groupby(group_columns)["action"].nunique()
        if not np.all(action_counts.to_numpy() == EXPECTED_ACTIONS):
            raise RuntimeError("Every intervention unit must have ten actions")
        if linear.duplicated(group_columns + ["action"]).any():
            raise RuntimeError("Duplicate source action rows")
        if not np.isfinite(
            linear[["true_cost", "predicted_cost"]].to_numpy()
        ).all():
            raise RuntimeError("Nonfinite source costs")
        source_state_counts = (
            linear.groupby("environment")["state_id"].nunique().to_dict()
        )
        if RUN_MODE == "smoke":
            keep_states = (
                linear[["environment", "state_id"]]
                .drop_duplicates()
                .groupby("environment", group_keys=False)
                .head(3)
            )
            keep_index = pd.MultiIndex.from_frame(keep_states)
            row_index = pd.MultiIndex.from_frame(
                linear[["environment", "state_id"]]
            )
            linear = linear.loc[row_index.isin(keep_index)].copy()
        print(
            f"Loaded {len(linear):,} linear-pose action rows from "
            f"{linear.groupby('environment')['state_id'].nunique().to_dict()}"
        )
    except Exception:
        record_failure("source_integrity")


In [ ]:
# Apply both exactly matched interventions and recompute physical planning outcomes.
if not PIPELINE_FAILED:
    try:
        unit_rows = []
        maximum_match_discrepancy = 0.0
        maximum_intact_cost_difference = 0.0
        maximum_fixed_points = 0
        source_unit_count = 0

        for keys, frame in linear.groupby(group_columns, sort=True):
            (
                environment,
                state_id,
                task_id,
                model,
                probe_seed,
                horizon,
            ) = keys
            frame = frame.sort_values("action")
            actions = frame["action"].to_numpy(dtype=int)
            if not np.array_equal(actions, np.arange(EXPECTED_ACTIONS)):
                raise RuntimeError(f"Noncanonical actions in unit {keys}")
            pose_dimension = 4 if environment == "PushT" else 2
            pose_columns = [
                f"predicted_pose_{index}"
                for index in range(pose_dimension)
            ]
            predicted_pose = frame[pose_columns].to_numpy(dtype=np.float64)
            true_cost = frame["true_cost"].to_numpy(dtype=np.float64)
            source_predicted_cost = frame[
                "predicted_cost"
            ].to_numpy(dtype=np.float64)
            if not np.isfinite(predicted_pose).all():
                raise RuntimeError(f"Nonfinite predicted pose in unit {keys}")
            task = task_lookup[(environment, int(task_id))]
            reconstructed_cost = decoded_task_cost(
                environment, predicted_pose, task
            )
            maximum_intact_cost_difference = max(
                maximum_intact_cost_difference,
                float(
                    np.max(
                        np.abs(
                            reconstructed_cost - source_predicted_cost
                        )
                    )
                ),
            )
            source_unit_count += 1

            for intervention_seed in INTERVENTION_SEEDS:
                derived_seed = stable_seed(
                    SEED,
                    intervention_seed,
                    environment,
                    int(state_id),
                    model,
                    int(probe_seed),
                    int(horizon),
                )
                for severity in SEVERITIES:
                    intervention = matched_pose_interventions(
                        predicted_pose,
                        severity,
                        derived_seed,
                    )
                    maximum_match_discrepancy = max(
                        maximum_match_discrepancy,
                        abs(
                            intervention["action_rms"]
                            - intervention["common_rms"]
                        ),
                    )
                    maximum_fixed_points = max(
                        maximum_fixed_points,
                        intervention["fixed_points"],
                    )
                    for intervention_name in [
                        "action_structure",
                        "common_mode",
                    ]:
                        intervened_pose = intervention[intervention_name]
                        predicted_cost = decoded_task_cost(
                            environment,
                            intervened_pose,
                            task,
                        )
                        ranking = ranking_metrics(
                            true_cost,
                            predicted_cost,
                        )
                        perturbation_rms = (
                            intervention["action_rms"]
                            if intervention_name == "action_structure"
                            else intervention["common_rms"]
                        )
                        unit_rows.append(
                            {
                                "environment": environment,
                                "state_id": int(state_id),
                                "task_id": int(task_id),
                                "model": model,
                                "probe_seed": int(probe_seed),
                                "horizon": int(horizon),
                                "intervention_seed": int(
                                    intervention_seed
                                ),
                                "intervention": intervention_name,
                                "severity": float(severity),
                                "pose_perturbation_rms": float(
                                    perturbation_rms
                                ),
                                "ordinary_cost_rmse": float(
                                    np.sqrt(
                                        np.mean(
                                            (
                                                predicted_cost - true_cost
                                            )
                                            ** 2
                                        )
                                    )
                                ),
                                **ranking,
                            }
                        )

        unit_metrics = pd.DataFrame(unit_rows)
        unit_metrics.to_csv(
            OUT / "intervention_unit_metrics.csv", index=False
        )
        expected_rows = (
            source_unit_count
            * len(INTERVENTION_SEEDS)
            * len(SEVERITIES)
            * 2
        )
        if len(unit_metrics) != expected_rows:
            raise AssertionError(
                f"Expected {expected_rows} intervention rows, "
                f"received {len(unit_metrics)}"
            )
        matched_integrity = {
            "source_stage3b_success": True,
            "source_state_clusters_before_smoke_filter": {
                key: int(value)
                for key, value in source_state_counts.items()
            },
            "source_units_analyzed": int(source_unit_count),
            "intervention_rows": int(len(unit_metrics)),
            "expected_actions_per_unit": EXPECTED_ACTIONS,
            "maximum_matched_pose_rms_discrepancy": float(
                maximum_match_discrepancy
            ),
            "match_tolerance": MATCH_TOLERANCE,
            "maximum_full_permutation_fixed_points": int(
                maximum_fixed_points
            ),
            "maximum_intact_decoded_cost_reconstruction_difference": float(
                maximum_intact_cost_difference
            ),
            "intact_reconstruction_tolerance": 1e-10,
            "nonfinite_co_primary_intervention_rows": int(
                np.sum(
                    ~np.isfinite(
                        unit_metrics[
                            [
                                "normalized_regret",
                                "weighted_pairwise_accuracy",
                            ]
                        ].to_numpy()
                    ).all(axis=1)
                )
            ),
        }
        matched_integrity["pass"] = bool(
            maximum_match_discrepancy <= MATCH_TOLERANCE
            and maximum_fixed_points == 0
            and maximum_intact_cost_difference <= 1e-10
        )
        write_json(
            OUT / "matched_error_integrity.json",
            matched_integrity,
        )
        if not matched_integrity["pass"]:
            raise AssertionError(
                f"Matched intervention integrity failed: {matched_integrity}"
            )
        print(
            f"Generated {len(unit_metrics):,} matched intervention units; "
            f"max match discrepancy={maximum_match_discrepancy:.3e}"
        )
    except Exception:
        record_failure("matched_interventions")


In [ ]:
# State-clustered summaries, full-severity co-primary tests, and frozen decision.
if not PIPELINE_FAILED:
    try:
        summary_rows = []
        summary_metrics = [
            "normalized_regret",
            "weighted_pairwise_accuracy",
            "top1_correct",
            "ordinary_cost_rmse",
            "normalized_margin_rmse",
        ]
        for environment in ["PushT", "Wall"]:
            for intervention_name in [
                "action_structure",
                "common_mode",
            ]:
                for severity in SEVERITIES:
                    selected = unit_metrics.loc[
                        unit_metrics["environment"].eq(environment)
                        & unit_metrics["intervention"].eq(
                            intervention_name
                        )
                        & unit_metrics["severity"].eq(severity)
                    ]
                    groups = selected["state_id"].to_numpy()
                    row = {
                        "environment": environment,
                        "intervention": intervention_name,
                        "severity": float(severity),
                        "num_rows": int(len(selected)),
                        "num_state_clusters": int(
                            selected["state_id"].nunique()
                        ),
                    }
                    for metric_index, metric in enumerate(summary_metrics):
                        estimate = bootstrap_mean(
                            selected[metric].to_numpy(),
                            groups,
                            BOOTSTRAP_REPS,
                            SEED
                            + 1000 * ["PushT", "Wall"].index(environment)
                            + 100 * [
                                "action_structure",
                                "common_mode",
                            ].index(intervention_name)
                            + 10 * SEVERITIES.index(severity)
                            + metric_index,
                        )
                        for key, value in estimate.items():
                            row[f"{metric}_{key}"] = value
                    summary_rows.append(row)
        intervention_summary = pd.DataFrame(summary_rows)
        intervention_summary.to_csv(
            OUT / "intervention_summary.csv", index=False
        )

        full = unit_metrics.loc[
            unit_metrics["severity"].eq(1.0)
        ].copy()
        pair_keys = [
            "environment",
            "state_id",
            "task_id",
            "model",
            "probe_seed",
            "horizon",
            "intervention_seed",
        ]
        full_pivot = full.pivot(
            index=pair_keys,
            columns="intervention",
            values=[
                "normalized_regret",
                "weighted_pairwise_accuracy",
                "top1_correct",
                "ordinary_cost_rmse",
                "normalized_margin_rmse",
                "pose_perturbation_rms",
            ],
        )
        full_pairs = full_pivot.reset_index()
        full_pairs.columns = [
            "_".join(str(item) for item in column if str(item))
            if isinstance(column, tuple)
            else str(column)
            for column in full_pairs.columns
        ]
        full_pairs["specific_regret_damage"] = (
            full_pairs["normalized_regret_action_structure"]
            - full_pairs["normalized_regret_common_mode"]
        )
        full_pairs["specific_ranking_damage"] = (
            full_pairs["weighted_pairwise_accuracy_common_mode"]
            - full_pairs[
                "weighted_pairwise_accuracy_action_structure"
            ]
        )
        full_pairs["specific_top1_damage"] = (
            full_pairs["top1_correct_common_mode"]
            - full_pairs["top1_correct_action_structure"]
        )
        full_pairs["task_margin_specificity"] = (
            full_pairs["normalized_margin_rmse_action_structure"]
            - full_pairs["normalized_margin_rmse_common_mode"]
        )
        full_pairs.to_csv(
            OUT / "full_severity_pairs.csv", index=False
        )
        finite_decision_mask = np.isfinite(
            full_pairs[
                [
                    "specific_regret_damage",
                    "specific_ranking_damage",
                ]
            ].to_numpy()
        ).all(axis=1)
        decision_pairs = full_pairs.loc[finite_decision_mask].copy()
        finite_cluster_counts = {
            environment: int(
                decision_pairs.loc[
                    decision_pairs["environment"].eq(environment),
                    "state_id",
                ].nunique()
            )
            for environment in ["PushT", "Wall"]
        }
        matched_integrity[
            "finite_full_severity_state_clusters"
        ] = finite_cluster_counts
        matched_integrity[
            "excluded_nondecision_full_severity_pairs"
        ] = int(len(full_pairs) - len(decision_pairs))
        matched_integrity[
            "common_finite_sample_for_both_co_primary_estimands"
        ] = True
        minimum_required_clusters = 2 if RUN_MODE == "smoke" else 20
        matched_integrity[
            "minimum_required_finite_state_clusters"
        ] = minimum_required_clusters
        matched_integrity["pass"] = bool(
            matched_integrity["pass"]
            and all(
                count >= minimum_required_clusters
                for count in finite_cluster_counts.values()
            )
        )
        write_json(
            OUT / "matched_error_integrity.json",
            matched_integrity,
        )
        if not matched_integrity["pass"]:
            raise AssertionError(
                f"Final matched intervention integrity failed: "
                f"{matched_integrity}"
            )

        slope_group_columns = [
            "environment",
            "state_id",
            "task_id",
            "model",
            "probe_seed",
            "horizon",
            "intervention_seed",
            "intervention",
        ]
        slope_rows = []
        for slope_keys, slope_frame in unit_metrics.groupby(
            slope_group_columns, sort=True
        ):
            slope_frame = slope_frame.sort_values("severity")
            severity_values = slope_frame["severity"].to_numpy()
            regret_values = slope_frame[
                "normalized_regret"
            ].to_numpy()
            ranking_values = slope_frame[
                "weighted_pairwise_accuracy"
            ].to_numpy()
            if not (
                np.isfinite(regret_values).all()
                and np.isfinite(ranking_values).all()
            ):
                continue
            slope_rows.append(
                {
                    **{
                        column: value
                        for column, value in zip(
                            slope_group_columns, slope_keys
                        )
                    },
                    "regret_slope": float(
                        np.polyfit(
                            severity_values, regret_values, 1
                        )[0]
                    ),
                    "ranking_slope": float(
                        np.polyfit(
                            severity_values, ranking_values, 1
                        )[0]
                    ),
                }
            )
        slope_units = pd.DataFrame(slope_rows)
        slope_pivot = slope_units.pivot(
            index=pair_keys,
            columns="intervention",
            values=["regret_slope", "ranking_slope"],
        ).reset_index()
        slope_pivot.columns = [
            "_".join(str(item) for item in column if str(item))
            if isinstance(column, tuple)
            else str(column)
            for column in slope_pivot.columns
        ]
        slope_pivot["specific_regret_slope"] = (
            slope_pivot["regret_slope_action_structure"]
            - slope_pivot["regret_slope_common_mode"]
        )
        slope_pivot["specific_ranking_slope"] = (
            slope_pivot["ranking_slope_common_mode"]
            - slope_pivot["ranking_slope_action_structure"]
        )
        slope_pivot.to_csv(
            OUT / "dose_response_slopes.csv", index=False
        )

        slope_comparisons = {}
        for environment in ["PushT", "Wall"]:
            selected = slope_pivot.loc[
                slope_pivot["environment"].eq(environment)
            ]
            groups = selected["state_id"].to_numpy()
            slope_comparisons[environment] = {
                "specific_regret_slope": bootstrap_mean(
                    selected["specific_regret_slope"].to_numpy(),
                    groups,
                    BOOTSTRAP_REPS,
                    SEED
                    + 7400
                    + ["PushT", "Wall"].index(environment),
                ),
                "specific_ranking_slope": bootstrap_mean(
                    selected["specific_ranking_slope"].to_numpy(),
                    groups,
                    BOOTSTRAP_REPS,
                    SEED
                    + 7500
                    + ["PushT", "Wall"].index(environment),
                ),
            }

        subgroup_rows = []
        subgroup_metrics = [
            "specific_regret_damage",
            "specific_ranking_damage",
            "specific_top1_damage",
            "task_margin_specificity",
        ]
        for (
            environment,
            model,
            horizon,
        ), subgroup in decision_pairs.groupby(
            ["environment", "model", "horizon"], sort=True
        ):
            row = {
                "environment": environment,
                "model": model,
                "horizon": int(horizon),
                "num_rows": int(len(subgroup)),
                "num_state_clusters": int(
                    subgroup["state_id"].nunique()
                ),
            }
            for metric_index, metric in enumerate(subgroup_metrics):
                estimate = bootstrap_mean(
                    subgroup[metric].to_numpy(),
                    subgroup["state_id"].to_numpy(),
                    BOOTSTRAP_REPS,
                    SEED
                    + 8000
                    + 1000 * ["PushT", "Wall"].index(environment)
                    + 100 * int(horizon)
                    + metric_index,
                )
                for key, value in estimate.items():
                    row[f"{metric}_{key}"] = value
            subgroup_rows.append(row)
        subgroup_specificity = pd.DataFrame(subgroup_rows)
        subgroup_specificity.to_csv(
            OUT / "subgroup_specificity.csv", index=False
        )
        subgroup_direction_consistency = {
            metric: {
                "positive_cells": int(
                    np.sum(
                        subgroup_specificity[
                            f"{metric}_estimate"
                        ].to_numpy()
                        > 0
                    )
                ),
                "total_cells": int(len(subgroup_specificity)),
                "confirmatory_gate": False,
            }
            for metric in subgroup_metrics
        }

        environment_comparisons = {}
        environment_gate_pass = {}
        for environment in ["PushT", "Wall"]:
            selected = decision_pairs.loc[
                decision_pairs["environment"].eq(environment)
            ]
            groups = selected["state_id"].to_numpy()
            regret_result = bootstrap_mean(
                selected["specific_regret_damage"].to_numpy(),
                groups,
                BOOTSTRAP_REPS,
                SEED + 7000 + ["PushT", "Wall"].index(environment),
            )
            ranking_result = bootstrap_mean(
                selected["specific_ranking_damage"].to_numpy(),
                groups,
                BOOTSTRAP_REPS,
                SEED + 7100 + ["PushT", "Wall"].index(environment),
            )
            top1_result = bootstrap_mean(
                selected["specific_top1_damage"].to_numpy(),
                groups,
                BOOTSTRAP_REPS,
                SEED + 7200 + ["PushT", "Wall"].index(environment),
            )
            margin_result = bootstrap_mean(
                selected["task_margin_specificity"].to_numpy(),
                groups,
                BOOTSTRAP_REPS,
                SEED + 7300 + ["PushT", "Wall"].index(environment),
            )
            environment_comparisons[environment] = {
                "specific_regret_damage": regret_result,
                "specific_ranking_damage": ranking_result,
                "specific_top1_damage": top1_result,
                "task_margin_specificity": margin_result,
            }
            environment_gate_pass[environment] = bool(
                regret_result["low"] > 0
                and ranking_result["low"] > 0
            )

        if not matched_integrity["pass"]:
            status = "INCONCLUSIVE"
        elif all(environment_gate_pass.values()):
            status = "CROSS_ENV_ACTION_STRUCTURE_CAUSAL_SIGNAL"
        elif any(environment_gate_pass.values()):
            status = "MIXED_ACTION_STRUCTURE_SIGNAL"
        else:
            status = "NO_ACTION_STRUCTURE_SPECIFICITY"

        stage4_decision = {
            "status": status,
            "environment_comparisons": environment_comparisons,
            "environment_gate_pass": environment_gate_pass,
            "dose_response_slope_comparisons": slope_comparisons,
            "descriptive_subgroup_direction_consistency": (
                subgroup_direction_consistency
            ),
            "integrity_pass": bool(matched_integrity["pass"]),
            "primary_gate": (
                "In both environments, state-clustered 95% bootstrap lower "
                "bounds must exceed zero for full-severity regret damage "
                "(action-structure minus common-mode) and ranking damage "
                "(common-mode minus action-structure), under exactly matched "
                "decoded-pose perturbation RMS."
            ),
            "claim_boundary": (
                "A pass identifies causal necessity of action-specific "
                "structure in decoded predictions for these simulator "
                "candidate sets. It does not identify the training mechanism, "
                "predict natural per-instance failures, or establish "
                "real-robot reliability."
            ),
        }
        write_json(OUT / "stage4_decision.json", stage4_decision)
        print(json.dumps(stage4_decision, indent=2))
    except Exception:
        record_failure("analysis_and_decision")


In [ ]:
# Publication-facing plots with state-clustered uncertainty.
if not PIPELINE_FAILED:
    try:
        colors = {
            "action_structure": "#d62728",
            "common_mode": "#1f77b4",
        }
        labels = {
            "action_structure": "action structure",
            "common_mode": "common mode",
        }

        def dose_response_plot(metric, ylabel, filename):
            figure, axes = plt.subplots(
                1, 2, figsize=(12, 4.5), sharey=True
            )
            for axis, environment in zip(axes, ["PushT", "Wall"]):
                for intervention_name in [
                    "action_structure",
                    "common_mode",
                ]:
                    selected = intervention_summary.loc[
                        intervention_summary["environment"].eq(environment)
                        & intervention_summary["intervention"].eq(
                            intervention_name
                        )
                    ].sort_values("severity")
                    estimate = selected[f"{metric}_estimate"].to_numpy()
                    low = selected[f"{metric}_low"].to_numpy()
                    high = selected[f"{metric}_high"].to_numpy()
                    axis.plot(
                        selected["severity"],
                        estimate,
                        marker="o",
                        linewidth=2,
                        color=colors[intervention_name],
                        label=labels[intervention_name],
                    )
                    axis.fill_between(
                        selected["severity"],
                        low,
                        high,
                        color=colors[intervention_name],
                        alpha=0.16,
                    )
                axis.set_title(environment)
                axis.set_xlabel("matched perturbation severity")
                axis.grid(alpha=0.2)
            axes[0].set_ylabel(ylabel)
            axes[1].legend(frameon=False)
            figure.suptitle(
                "Matched decoded-pose interventions on untouched final-test states"
            )
            figure.tight_layout()
            figure.savefig(PLOT_DIR / filename, dpi=180)
            plt.close(figure)

        dose_response_plot(
            "normalized_regret",
            "physical normalized regret",
            "matched_regret_dose_response.png",
        )
        dose_response_plot(
            "weighted_pairwise_accuracy",
            "margin-weighted pairwise accuracy",
            "matched_ranking_dose_response.png",
        )

        figure, axes = plt.subplots(1, 2, figsize=(10, 4.5))
        for axis, (metric, title) in zip(
            axes,
            [
                ("specific_regret_damage", "specific regret damage"),
                ("specific_ranking_damage", "specific ranking damage"),
            ],
        ):
            estimates = []
            lows = []
            highs = []
            for environment in ["PushT", "Wall"]:
                result = environment_comparisons[environment][metric]
                estimates.append(result["estimate"])
                lows.append(result["low"])
                highs.append(result["high"])
            estimates = np.asarray(estimates)
            axis.bar(
                ["PushT", "Wall"],
                estimates,
                color=["#9467bd", "#2ca02c"],
            )
            axis.errorbar(
                [0, 1],
                estimates,
                yerr=[
                    estimates - np.asarray(lows),
                    np.asarray(highs) - estimates,
                ],
                fmt="none",
                color="black",
                capsize=4,
            )
            axis.axhline(0, color="black", linewidth=1)
            axis.set_title(title)
            axis.grid(axis="y", alpha=0.2)
        figure.suptitle("Full-severity action-structure specificity")
        figure.tight_layout()
        figure.savefig(
            PLOT_DIR / "full_severity_specificity.png", dpi=180
        )
        plt.close(figure)
    except Exception:
        record_failure("plots")


In [ ]:
# Bundle all evidence and automatically download the result ZIP.
def package_results():
    include = [
        "FAILURE_TRACE.txt",
        "stage4_config.json",
        "source_bundle_manifest.json",
        "intervention_unit_metrics.csv",
        "intervention_summary.csv",
        "full_severity_pairs.csv",
        "dose_response_slopes.csv",
        "subgroup_specificity.csv",
        "matched_error_integrity.json",
        "stage4_decision.json",
        "plots/matched_regret_dose_response.png",
        "plots/matched_ranking_dose_response.png",
        "plots/full_severity_specificity.png",
    ]
    existing = [name for name in include if (OUT / name).is_file()]
    manifest = {
        "archive": str(RESULT_ZIP),
        "included": existing + ["result_zip_manifest.json"],
        "source_stage3b_archive_excluded": True,
        "intermediate_excluded": True,
    }
    write_json(OUT / "result_zip_manifest.json", manifest)
    if RESULT_ZIP.exists():
        RESULT_ZIP.unlink()
    with zipfile.ZipFile(
        RESULT_ZIP, "w", compression=zipfile.ZIP_DEFLATED
    ) as handle:
        for name in manifest["included"]:
            handle.write(OUT / name, arcname=name)
    return RESULT_ZIP


try:
    archive = package_results()
    print(f"Created {archive} ({archive.stat().st_size / 1e6:.2f} MB)")
    if DOWNLOAD_RESULTS:
        from google.colab import files

        files.download(str(archive))
except Exception:
    record_failure("package_and_download")
    archive = package_results()
    print(f"Failure bundle created at {archive}")


## Interpretation boundary

A cross-environment pass supports a narrow causal statement: under fixed
simulator states and candidate sets, action-specific structure in the decoded
predicted consequences is more important for planning than an equally large
shared decoded-state error.

It does **not** show that the world models learned that structure by a
particular mechanism, that the Stage 3B error score predicts natural
per-instance failures, or that the result transfers to physical robots.
